# SQL Database Parsing

Databases are one of the most important sources of structured data in real applications.

Unlike files such as PDF or JSON, database data is usually spread across tables and connected through relationships.

In this notebook, I’m learning how to create a small SQLite database, inspect its structure, and convert database information into LangChain `Document` objects.

I’m also looking at how table-level information and relationships between tables can be preserved for a RAG pipeline.

The main flow I’m following is:

**Database → Inspect Schema → Extract Data → Create Documents → Preserve Relationships and Metadata**

## Setting Up the Database

I’m using SQLite for this notebook because it is simple to create and does not require a separate database server.

I’m keeping the database file inside my project so I can experiment with queries and document processing easily.

In [17]:
# create sample SQLite Database
import sqlite3
import os
from pathlib import Path
import sqlite3

DB_PATH = Path("E:/RAG/data/course_samples/databases/company.db")

DB_PATH.parent.mkdir(parents=True, exist_ok=True)

print(DB_PATH.resolve())

E:\RAG\data\course_samples\databases\company.db


## Creating a Sample Database

I’m creating a small company database with two tables:

- `employees`
- `projects`

The idea is to have enough structure to practice both normal table extraction and relationships between tables.

In [18]:
# create sample database
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

## Creating the Employees Table

The `employees` table contains basic information about each employee.

Each employee has:

- ID
- Name
- Role
- Department
- Salary

In [19]:
# Create tables
cursor.execute("""
    CREATE TABLE IF NOT EXISTS employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        role TEXT,
        department TEXT,
        salary REAL
    )
""")

## Creating the Projects Table

The second table stores project information.

Each project has:

- ID
- Project name
- Status
- Budget
- `lead_id`

The `lead_id` connects a project with an employee.

This relationship will be important later when I create relationship-based documents.

In [20]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS projects (
        id INTEGER PRIMARY KEY,
        name TEXT,
        status TEXT,
        budget REAL,
        lead_id INTEGER
    )
""")

## Adding Sample Data

Now I’m inserting some sample employees and projects into the database.

I’m keeping the data small because the goal here is to understand the database structure and the ingestion process, not to work with a large dataset yet.

In [21]:
employees = [
    (1, "John Doe", "Senior Developer", "Engineering", 95000),
    (2, "Jane Smith", "Data Scientist", "Analytics", 105000),
    (3, "Mike Johnson", "Product Manager", "Product", 110000),
    (4, "Sarah Williams", "DevOps Engineer", "Engineering", 98000)
]

projects = [
    (1, "RAG Implementation", "Active", 150000, 1),
    (2, "Data Pipeline", "Completed", 80000, 2),
    (3, "Customer Portal", "Planning", 200000, 3),
    (4, "ML Platform", "Active", 250000, 2)
]

cursor.executemany(
    "INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?, ?)",
    employees
)

cursor.executemany(
    "INSERT OR REPLACE INTO projects VALUES (?, ?, ?, ?, ?)",
    projects
)

conn.commit()

## Checking the Database

Before moving to the LangChain part, I want to make sure the database was created correctly.

I’m running a simple query on the employees table to check that the data is actually there.

In [22]:
print("Employees:")
cursor.execute("SELECT * FROM employees")
print(cursor.fetchall())

print("\nProjects:")
cursor.execute("SELECT * FROM projects")
print(cursor.fetchall())

Employees:
[(1, 'John Doe', 'Senior Developer', 'Engineering', 95000.0), (2, 'Jane Smith', 'Data Scientist', 'Analytics', 105000.0), (3, 'Mike Johnson', 'Product Manager', 'Product', 110000.0), (4, 'Sarah Williams', 'DevOps Engineer', 'Engineering', 98000.0)]

Projects:
[(1, 'RAG Implementation', 'Active', 150000.0, 1), (2, 'Data Pipeline', 'Completed', 80000.0, 2), (3, 'Customer Portal', 'Planning', 200000.0, 3), (4, 'ML Platform', 'Active', 250000.0, 2)]


In [23]:
cursor.execute("SELECT * FROM employees")

In [24]:
cursor.fetchall()

[(1, 'John Doe', 'Senior Developer', 'Engineering', 95000.0),
 (2, 'Jane Smith', 'Data Scientist', 'Analytics', 105000.0),
 (3, 'Mike Johnson', 'Product Manager', 'Product', 110000.0),
 (4, 'Sarah Williams', 'DevOps Engineer', 'Engineering', 98000.0)]

In [25]:
conn.commit()
conn.close()

### What I noticed

The database now contains two related tables.

This is different from a normal file because the data is stored in a structured system where tables can be connected to each other.

That relationship becomes important when I prepare database information for retrieval.

## Database Content Extraction

Now I’m moving from basic SQLite operations to LangChain.

I want to inspect the database schema and understand what information LangChain can read from it.

I’m using:

- `SQLDatabase`
- `SQLDatabaseLoader`

The first one helps me inspect the database structure, while the second can be used for loading query results into documents.

In [26]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

## 1. Using SQLDatabase

I’m connecting LangChain to the SQLite database.

The first thing I want to inspect is:

- Which tables exist
- What their schema looks like
- What sample rows are available

This helps me understand what information is available before I start creating documents.

In [27]:
db = SQLDatabase.from_uri(
    "sqlite:///E:/RAG/data/course_samples/databases/company.db"
)

print(f"Tables: {db.get_usable_table_names()}")

print("\nTable DDL:")
print(db.get_table_info())

Tables: ['employees', 'projects']

Table DDL:

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	John Doe	Senior Developer	Engineering	95000.0
2	Jane Smith	Data Scientist	Analytics	105000.0
3	Mike Johnson	Product Manager	Product	110000.0
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	RAG Implementation	Active	150000.0	1
2	Data Pipeline	Completed	80000.0	2
3	Customer Portal	Planning	200000.0	3
*/


### What I noticed

`SQLDatabase` gives me more than just the rows.

I can also see the table definitions and sample records.

This is useful for a RAG system because database structure itself can be important when deciding how the data should be retrieved.

## Why Database Schema Matters

A database is not just a collection of rows.

The schema tells me:

- What tables exist
- What columns they contain
- What type of data is stored
- How different pieces of data are related

For example, `projects.lead_id` connects project records with employees.

If I ignore that relationship, I may lose useful context when creating documents.

## 2. Custom SQL Processing

Now I want more control over how database information becomes a document.

Instead of simply loading query results, I’m going to create documents myself.

For each table, I’ll keep:

- Table name
- Column names
- Number of records
- Sample records

I’ll also create a separate document for the relationship between employees and projects.

This gives me more control over what context is available during retrieval.

In [28]:
from typing import List
from langchain_core.documents import Document

## Creating the SQL-to-Document Processor

This function reads the database and converts it into LangChain documents.

The first part creates one overview document for each table.

The second part creates a relationship document by joining the employees and projects tables.

This lets me preserve both table information and cross-table context.

In [29]:
def sql_to_documents(db_path: str) -> List[Document]:
    """Convert SQL database information into documents with context."""

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    documents = []

    # Find all tables
    cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type='table';
    """)

    tables = cursor.fetchall()

    # Create one document for each table
    for table in tables:

        table_name = table[0]

        # Get table schema
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()

        column_names = [col[1] for col in columns]

        # Get table data
        cursor.execute(f"SELECT * FROM {table_name}")
        rows = cursor.fetchall()

        # Create table overview
        table_content = f"Table: {table_name}\n"
        table_content += f"Columns: {', '.join(column_names)}\n"
        table_content += f"Total Records: {len(rows)}\n\n"

        table_content += "Sample Records:\n"

        for row in rows[:5]:
            record = dict(zip(column_names, row))
            table_content += f"{record}\n"

        doc = Document(
            page_content=table_content,
            metadata={
                "source": db_path,
                "table_name": table_name,
                "num_records": len(rows),
                "data_type": "sql_table"
            }
        )

        documents.append(doc)

    # Create a relationship document
    cursor.execute("""
        SELECT
            e.name,
            e.role,
            p.name AS project_name,
            p.status
        FROM employees e
        JOIN projects p
            ON e.id = p.lead_id
    """)

    relationships = cursor.fetchall()

    rel_content = "Employee-Project Relationships:\n\n"

    for rel in relationships:
        rel_content += (
            f"{rel[0]} ({rel[1]}) leads "
            f"{rel[2]} - Status: {rel[3]}\n"
        )

    rel_doc = Document(
        page_content=rel_content,
        metadata={
            "source": db_path,
            "data_type": "sql_relationships",
            "query": "employee_project_join"
        }
    )

    documents.append(rel_doc)

    conn.close()

    return documents

## Processing the Database

Now I’m running the complete SQL-to-document pipeline.

I expect:

- One document for the `employees` table
- One document for the `projects` table
- One document describing employee-project relationships

This is where the database structure is being converted into content that a RAG pipeline can work with.

In [30]:
sql_documents = sql_to_documents(
    "E:/RAG/data/course_samples/databases/company.db"
)

print(f"Created {len(sql_documents)} documents")

Created 3 documents


## Looking at the Generated Documents

I’m printing the documents here so I can see exactly what information has been preserved.

I’m checking both the content and the metadata because both will matter later during retrieval.

In [31]:
for i, doc in enumerate(sql_documents):

    print(f"\nDocument {i + 1}")
    print("=" * 50)

    print(doc.page_content)

    print("\nMetadata:")
    print(doc.metadata)


Document 1
Table: employees
Columns: id, name, role, department, salary
Total Records: 4

Sample Records:
{'id': 1, 'name': 'John Doe', 'role': 'Senior Developer', 'department': 'Engineering', 'salary': 95000.0}
{'id': 2, 'name': 'Jane Smith', 'role': 'Data Scientist', 'department': 'Analytics', 'salary': 105000.0}
{'id': 3, 'name': 'Mike Johnson', 'role': 'Product Manager', 'department': 'Product', 'salary': 110000.0}
{'id': 4, 'name': 'Sarah Williams', 'role': 'DevOps Engineer', 'department': 'Engineering', 'salary': 98000.0}


Metadata:
{'source': 'E:/RAG/data/course_samples/databases/company.db', 'table_name': 'employees', 'num_records': 4, 'data_type': 'sql_table'}

Document 2
Table: projects
Columns: id, name, status, budget, lead_id
Total Records: 4

Sample Records:
{'id': 1, 'name': 'RAG Implementation', 'status': 'Active', 'budget': 150000.0, 'lead_id': 1}
{'id': 2, 'name': 'Data Pipeline', 'status': 'Completed', 'budget': 80000.0, 'lead_id': 2}
{'id': 3, 'name': 'Customer Po

### What I noticed

The database has now been converted into different types of documents.

The table documents keep schema and sample record information.

The relationship document keeps information that comes from joining multiple tables.

This is useful because some questions need a single table, while other questions depend on relationships between tables.

## Why Create Different Document Types?

I could put everything into one huge document, but that would mix many different types of information together.

Instead, I’m keeping:

**Table documents**
→ Structure and sample records

**Relationship documents**
→ Connections between related tables

This makes the information more focused and can help retrieval return more relevant context.

## Why This Matters in RAG

Imagine I’m building a company knowledge assistant connected to an internal database.

A user could ask:

- Who is the Data Scientist?
- What is Jane Smith's salary?
- Which projects are currently active?
- Who is leading the RAG Implementation project?
- Which employee is working on the ML Platform?
- What projects are connected to Jane Smith?

Some questions only need one table.

Other questions need information from multiple tables.

That is why I need to think about both the database structure and the relationships when preparing data for RAG.

## My Takeaway

Databases are different from files because the information is usually organized across multiple tables and relationships.

In this notebook, I learned how to:

- Create a SQLite database
- Create and populate tables
- Inspect database schema with `SQLDatabase`
- Read table information
- Convert tables into LangChain documents
- Preserve metadata
- Create relationship-based documents using SQL joins

The main flow I’m taking from this notebook is:

**Database → Inspect Schema → Extract Data → Create Table Documents → Preserve Relationships → Prepare for RAG**

The important lesson for me is that database ingestion is not just about copying rows into documents.

I also need to preserve the structure and relationships that give the data its meaning.